# Meta Motivo benchmarking using HumEnv

This notebook shows how to evaluate a Meta Motivo model using the benchmark proposed in HumEnv. It assumes that motions for tracking and poses for goal reaching have been processed by following the [instructions](https://github.com/facebookresearch/humenv/tree/main/data_preparation) in HumEnv and are available in the folder `MOTIONS_BASE_PATH`.

In [1]:
import os
os.environ["MUJOCO_GL"] = os.environ.get("MUJOCO_GL", "egl")
os.environ["PYOPENGL_PLATFORM"] = os.environ["MUJOCO_GL"]

from pathlib import Path
import mediapy as media
import numpy as np
from gymnasium.wrappers import FlattenObservation, TransformObservation
from functools import reduce 

# humenv
import humenv
from humenv import make_humenv

from humenv.env import make_from_name
from humenv import rewards as humenv_rewards

In [2]:
import os
import sys
sys.path.append("..")
os.environ['CUDA_VISIBLE_DEVICES']='3'

from metamotivo.fb_cpr.huggingface import FBcprModel
from metamotivo.wrappers.humenvbench import RewardWrapper, TrackingWrapper, GoalWrapper
from metamotivo.buffers.buffers import DictBuffer
from huggingface_hub import hf_hub_download
import h5py
import json
import numpy as np
from humenv import STANDARD_TASKS
from humenv.bench import (
    RewardEvaluation,
    GoalEvaluation,
)

# paths where to find the output of HumEnv's data preparation scripts
MOTIONS_BASE_PATH = "/home/jovyan/Metamotivo/data/humenv_amass"
MOTIONS_TRACKING = "/home/jovyan/Metamotivo/data/test_train_split/large1_small1_train_0.1.txt"
GOAL_POSES = "/home/jovyan/Metamotivo/data/goal_poses/goals.json"

# load the goal poses into a dictionary
with open(GOAL_POSES, "r") as json_file:
    GOAL_DICT = json.load(json_file)
GOAL_DICT = {k: np.array(v["observation"]) for k,v in GOAL_DICT.items()}

Load inference buffer.

In [3]:
buffer_path = hf_hub_download(
        repo_id="facebook/metamotivo-S-1",
        filename="data/buffer_inference_500000.hdf5",
        repo_type="model",
        local_dir="metamotivo-S-1-datasets",
    )
hf = h5py.File(buffer_path, "r")
data = {k: v[:] for k, v in hf.items()}
buffer = DictBuffer(capacity=data["qpos"].shape[0], device="cuda")
buffer.extend(data)

Load model and prepare it for inference.

In [ ]:
from src.huggingface import MotionDiffuseFBcprModel

device = "cuda"  # it is normally faster to evaluate on cpu since tracking is parallelized
model = FBcprModel.from_pretrained("facebook/metamotivo-S-1").to(device)
# model = MotionDiffuseFBcprModel.from_pretrained("facebook/metamotivo-S-1")

base_model = RewardWrapper(
        model=model,
        inference_dataset=buffer,
        num_samples_per_inference=100_000,
        inference_function="reward_wr_inference",
        max_workers=80,
    )
base_model = GoalWrapper(model=base_model)
base_model = TrackingWrapper(model=base_model)

In [17]:
reward_eval = RewardEvaluation(
        tasks=['move-ego-0-0', 'jump-2'],
        env_kwargs={
            "state_init": "Fall",
        }, 
        num_contexts=1,
        num_envs=1,
        num_episodes=100,
        vectorization_mode='sync'
    )

reward_metrics = reward_eval.run(agent=base_model)
print(reward_metrics)

task move-ego-0-0 (inference):   0%|          | 0/2 [00:00<?, ?it/s]

{'move-ego-0-0': {'reward': [np.float64(270.4616546675615), np.float64(268.25495622283364), np.float64(266.14326156922846), np.float64(272.3730997748235), np.float64(272.35307555739), np.float64(271.80721379874166), np.float64(271.7807821585577), np.float64(271.2598243252475), np.float64(270.2797753743851), np.float64(262.7103658075076), np.float64(270.05085683514824), np.float64(272.1792857964894), np.float64(270.98074305240385), np.float64(268.0819258916858), np.float64(266.48802227337313), np.float64(275.9415949361676), np.float64(270.11422450782226), np.float64(274.4568087065494), np.float64(253.5422567320089), np.float64(272.3546324107145), np.float64(266.4208242882715), np.float64(265.07968154899237), np.float64(273.3805842824524), np.float64(261.6763580781132), np.float64(270.0359227418423), np.float64(271.6527186037136), np.float64(264.7804850214462), np.float64(270.94596716441305), np.float64(270.4325964067333), np.float64(265.06478608025515), np.float64(268.52129183577387), n

In [18]:
from functools import reduce 
reduce(lambda x, y: x + y, reward_metrics['jump-2']['reward']) / len(reward_metrics['jump-2']['reward'])

37.49871224876909

Humenv provides 3 evaluation protocols:
- reward based,
- goal based,
- tracking

In [ ]:
reward_eval = RewardEvaluation(
        tasks=STANDARD_TASKS,  # all the 45 tasks used in the paper
        env_kwargs={"state_init": "Fall"},
        num_contexts=1,
        num_envs=50,
        num_episodes=100,
    )

reward_metrics = reward_eval.run(agent=base_model)
print(reward_metrics)

r = np.array([m['reward'] for m in reward_metrics.values()])
print(f"reward averaged across {r.shape[0]} tasks: {r.mean()}")

task move-ego--90-2 (inference):  11%|█         | 5/45 [12:56<1:42:16, 153.41s/it]  

In [ ]:
goal_eval = GoalEvaluation(
    goals=GOAL_DICT,
    env_kwargs={"state_init": "Fall"},
    num_contexts=1,
    num_envs=50,
    num_episodes=100,
)

goal_metrics = goal_eval.run(agent=model)
print(goal_metrics)

for k in ['success', 'proximity']:
    r = np.array([m[k] for m in goal_metrics.values()])
    print(f"goal {k} averaged across {r.shape[0]} poses: {r.mean()}")

## Motions Evaluation

In [4]:
import sys
sys.path.append("../")

from utils.text_tracking_evaluation import TextTrackingEvaluation

# MOTIONS_BASE_PATH = "/home/jovyan/bobrin/TextMetamotivo/notebooks/amass_motions_annotated_nonsmoothedZ.hdf5"
MOTIONS_BASE_PATH = "/home/jovyan/Metamotivo/TextMetamotivo/notebooks/Full_dataset_motions_annotated_nonsmoothedZ.hdf5"

tracking_eval = TextTrackingEvaluation(
    motions=MOTIONS_BASE_PATH,
    keys=["observation", "text", "qpos", "qvel", "motion_id", "text_embedding", "fb_traj_embedding"],
    env_kwargs={"state_init": "Default"},
    num_envs=50,
)

tracking_metrics = tracking_eval.run_text(agent=model)
print(tracking_metrics)

KeyboardInterrupt: 

In [13]:
for k in ['success_phc_linf', 'emd']:
    r = np.array([m[k] for m in tracking_metrics.values()])
    print(f"tracking {k} averaged across {r.shape[0]} motions: {r.mean()}")

tracking success_phc_linf averaged across 8751 motions: 0.8504170951891212
tracking emd averaged across 8751 motions: 1.3348004169208476
